In [15]:
import pandas as pd
import requests
import os
from datetime import datetime, timedelta
import io
import zipfile

def download_and_save_petrinex(months_back=24):
    """
    Récupère les données Petrinex pour les X derniers mois.
    Structure de l'API : ZIP contenant un ZIP contenant un CSV.
    """
    base_url = "https://www.petrinex.gov.ab.ca/publicdata/API/Files/AB/Vol/{}/CSV"
    data_dir = "data/raw"
    os.makedirs(data_dir, exist_ok=True)

    current_date = datetime.now()
    count = 0
    month_offset = 2 # Délai habituel de publication des données

    all_summaries = []

    print(f"Début de la récupération pour {months_back} mois...")

    while count < months_back:
        target_date = current_date - timedelta(days=30 * (month_offset + count))
        month_str = target_date.strftime("%Y-%m")
        url = base_url.format(month_str)
        file_path = os.path.join(data_dir, f"petrinex_{month_str.replace('-', '_')}.csv")

        print(f"[{count+1}/{months_back}] Tentative pour {month_str}...")

        try:
            response = requests.get(url, timeout=120, stream=True)
            if response.status_code == 200:
                content = response.content
                with zipfile.ZipFile(io.BytesIO(content)) as z1:
                    inner_zip_name = z1.namelist()[0]
                    with z1.open(inner_zip_name) as f1:
                        with zipfile.ZipFile(io.BytesIO(f1.read())) as z2:
                            csv_filename = z2.namelist()[0]
                            with z2.open(csv_filename) as f2:
                                # Charger le CSV
                                df = pd.read_csv(f2, encoding="ISO-8859-1", low_memory=False)

                                # Sauvegarder localement
                                df.to_csv(file_path, index=False)

                                # Mappage des colonnes selon la structure réelle observée
                                # OperatorID -> OperatorBAID
                                # UWI -> FromToIDIdentifier (ou ReportingFacilityIdentifier selon le contexte)
                                # ProductionMonth -> ProductionMonth
                                # ProductType -> ProductID
                                # ActivityType -> ActivityID
                                # VolumeReported -> Volume

                                mapping = {
                                    'OperatorBAID': 'OperatorID',
                                    'FromToIDIdentifier': 'UWI',
                                    'ProductionMonth': 'ProductionMonth',
                                    'ProductID': 'ProductType',
                                    'ActivityID': 'ActivityType',
                                    'Volume': 'VolumeReported'
                                }

                                # Sélectionner et renommer
                                existing_cols = [c for c in mapping.keys() if c in df.columns]
                                summary = df[existing_cols].rename(columns=mapping)

                                if 'VolumeReported' in summary.columns:
                                    summary['VolumeReported'] = pd.to_numeric(summary['VolumeReported'], errors='coerce')

                                all_summaries.append(summary)
                                print(f" -> Succès : {len(df)} lignes sauvegardées.")
                                count += 1
            else:
                print(f" -> Non disponible (Code {response.status_code})")
        except Exception as e:
            print(f" -> Erreur : {e}")

        month_offset += 1
        if month_offset > 60: break

    if not all_summaries:
        return None

    return pd.concat(all_summaries, ignore_index=True)

def analyze_data(df):
    if df is None:
        print("Aucune donnée à analyser.")
        return

    print("\n" + "="*60)
    print("RAPPORT D'ANALYSE PETRINEX (Données agrégées)")
    print("="*60)

    print(f"\nNombre total de lignes analysées : {len(df)}")

    print("\n--- Statistiques sur les Volumes ---")
    if 'VolumeReported' in df.columns:
        print(df['VolumeReported'].describe())

    print("\n--- Top 10 des Types de Produits ---")
    if 'ProductType' in df.columns:
        print(df['ProductType'].value_counts().head(10))

    print("\n--- Top 10 des Types d'Activité ---")
    if 'ActivityType' in df.columns:
        print(df['ActivityType'].value_counts().head(10))

    print("\n--- Top 10 des Opérateurs par Volume Cumulé ---")
    if 'OperatorID' in df.columns and 'VolumeReported' in df.columns:
        top_ops = df.groupby('OperatorID')['VolumeReported'].sum().sort_values(ascending=False).head(10)
        print(top_ops)

if __name__ == "__main__":
    # Pour 24 mois, changez à 24. Ici 3 pour le test.
    final_df = download_and_save_petrinex(3)
    analyze_data(final_df)

Début de la récupération pour 3 mois...
[1/3] Tentative pour 2026-03...
 -> Succès : 548802 lignes sauvegardées.
[2/3] Tentative pour 2026-01...
 -> Succès : 550540 lignes sauvegardées.
[3/3] Tentative pour 2025-11...
 -> Succès : 553399 lignes sauvegardées.

RAPPORT D'ANALYSE PETRINEX (Données agrégées)

Nombre total de lignes analysées : 1652741

--- Statistiques sur les Volumes ---
count    1.575609e+06
mean     1.033191e+03
std      1.821217e+04
min     -1.116784e+05
25%      1.100000e+00
50%      1.130000e+01
75%      7.930000e+01
max      3.585222e+06
Name: VolumeReported, dtype: float64

--- Top 10 des Types de Produits ---
ProductType
GAS      748330
WATER    447014
OIL      305969
COND      15020
C5-SP      9549
STEAM      8678
C3-MX      5826
C2-MX      5523
C5-MX      5079
C4-MX      4933
Name: count, dtype: int64

--- Top 10 des Types d'Activité ---
ActivityType
PROD       730088
DISP       141320
FUEL       113389
REC        100562
DIFF        91185
VENT        80974
SHUTI